# German Declension Trainer — Level 1: Article + Nomen (All 4 Cases)

**How to use this notebook:**
1. Run **Cell 1** (setup & vocabulary database) once by pressing `Shift + Enter`.
2. Run **Cell 2** to start an interactive session (type `q` to quit anytime, or `stats` to see weak spots).
3. Easily expand `VOCAB` in Cell 1 with nouns from your **Uni Bonn** course or **Instagram** saves!

In [ ]:
import random

# Format: (gender, singular, plural, n_deklination_singular_ending, n_deklination_genitive_ending)
# Note: Standard nouns have "" for n_deklination. Masculine weak nouns (n-Deklination) take -n/-en in Akk/Dat/Gen.
VOCAB = [
    ("m", "Tisch", "Tische", "", ""),
    ("m", "Gedanke", "Gedanken", "n", "ns"),       # n-Deklination + Genitiv -ns
    ("m", "Student", "Studenten", "en", "en"),     # n-Deklination
    ("m", "Vorschlag", "Vorschläge", "", ""),
    ("m", "Alltag", "Alltage", "", ""),
    ("f", "Entscheidung", "Entscheidungen", "", ""),
    ("f", "Erfahrung", "Erfahrungen", "", ""),
    ("f", "Möglichkeit", "Möglichkeiten", "", ""),
    ("f", "Woche", "Wochen", "", ""),
    ("f", "Sprache", "Sprachen", "", ""),
    ("n", "Problem", "Probleme", "", ""),
    ("n", "Beispiel", "Beispiele", "", ""),
    ("n", "Ergebnis", "Ergebnisse", "", ""),
    ("n", "Ziel", "Ziele", "", ""),
    ("n", "Jahr", "Jahre", "", ""),
]

ARTICLES = {
    "bestimmt": {
        "Nominativ": {"m": "der", "f": "die", "n": "das", "pl": "die"},
        "Akkusativ": {"m": "den", "f": "die", "n": "das", "pl": "die"},
        "Dativ":     {"m": "dem", "f": "der", "n": "dem", "pl": "den"},
        "Genitiv":   {"m": "des", "f": "der", "n": "des", "pl": "der"},
    },
    "unbestimmt": {
        "Nominativ": {"m": "ein",   "f": "eine",  "n": "ein",   "pl": "keine"},
        "Akkusativ": {"m": "einen", "f": "eine",  "n": "ein",   "pl": "keine"},
        "Dativ":     {"m": "einem", "f": "einer", "n": "einem", "pl": "keinen"},
        "Genitiv":   {"m": "eines", "f": "einer", "n": "eines", "pl": "keiner"},
    },
}

CASE_TRIGGERS = {
    "Nominativ": ["Das Subjekt:", "Dort steht/sind", "Es geht um das Thema:"],
    "Akkusativ": ["Ich sehe / brauche", "Ohne", "Für", "Es geht um"],
    "Dativ":     ["Mit", "Nach", "Aus", "Bei", "Gegenüber"],
    "Genitiv":   ["Trotz", "Wegen", "Während", "Innerhalb", "Aufgrund"],
}

def inflect_noun(gender, singular, plural, n_end, n_gen, case, is_plural):
    if is_plural:
        # Dativ Plural always adds -n unless plural already ends in -n or -s
        if case == "Dativ" and not plural.endswith(("n", "s")):
            return plural + "n"
        return plural

    # Masculine n-Deklination (e.g., der Student -> den Studenten)
    if gender == "m" and n_end:
        if case == "Genitiv":
            return singular + (n_gen if n_gen else n_end)
        if case in ("Akkusativ", "Dativ"):
            return singular + n_end
        return singular

    # Standard Genitiv Singular for masculine & neuter (-s or -es)
    if case == "Genitiv" and gender in ("m", "n"):
        if singular.endswith(("s", "ß", "x", "z", "sch")) or len(singular) <= 5:
            return singular + "es"
        return singular + "s"

    return singular

def generate_item(filter_case=None):
    gender, sg, pl, n_end, n_gen = random.choice(VOCAB)
    is_plural = random.choice([False, False, True])  # ~33% plural practice
    slot = "pl" if is_plural else gender
    case = filter_case if filter_case else random.choice(list(ARTICLES["bestimmt"].keys()))
    art_type = random.choice(["bestimmt", "unbestimmt"])
    trigger = random.choice(CASE_TRIGGERS[case])

    article = ARTICLES[art_type][case][slot]
    noun_form = inflect_noun(gender, sg, pl, n_end, n_gen, case, is_plural)
    answer = f"{article} {noun_form}"

    # Hint tag
    art_label = "bestimmt" if art_type == "bestimmt" else ("negativ (kein-)" if is_plural else "unbestimmt")
    num_label = f"Plural (Sg: {sg})" if is_plural else f"Singular ({'der' if gender=='m' else 'die' if gender=='f' else 'das'} {sg})"
    prompt = f"[{case} | {art_label} | {num_label}]  →  {trigger} ___"
    return prompt, answer, case, slot

print("Setup complete! Run the next cell to start drilling.")

In [ ]:
# Interactive Drill Loop
# Optional: set FOCUS_CASE = "Genitiv" (or "Nominativ", "Akkusativ", "Dativ") to drill one case only, or None for all 4.
FOCUS_CASE = None

score = 0
total = 0
mistakes = []

print("Type 'q' to finish and see your summary.\n" + "-" * 55)

while True:
    prompt, expected, case, slot = generate_item(FOCUS_CASE)
    print(f"\n{prompt}")
    user_ans = input("Your answer: ").strip()

    if user_ans.lower() in ("q", "quit", "exit"):
        break

    total += 1
    if user_ans == expected:
        score += 1
        print(f"✅ Richtig! ({score}/{total})")
    else:
        mistakes.append((prompt, user_ans, expected))
        print(f"❌ Falsch! Dein Input: '{user_ans}'  →  Richtig: '{expected}'")

print("\n" + "=" * 55)
if total > 0:
    print(f"Session finished: {score}/{total} correct ({round(score/total*100)}%)")
    if mistakes:
        print("\nReview these slips:")
        for p, u, e in mistakes:
            print(f"  • {p}\n    You wrote: {u}  |  Correct: {e}")
else:
    print("Session ended.")